# FewShotPainAdaptation LOSO Hyperparameter Search on Google Colab

This notebook runs a persistent Optuna search over the current CrossMod CAN learned-prototype-memory LOSO pipeline. Each trial evaluates the selected configuration over `N_LOSO_STEPS` held-out subjects and maximizes source-subject prototype vote accuracy.

Artifacts are written to Google Drive after every trial: the Optuna SQLite study, full trial payloads, trial hyperparameters, a CSV/JSON trial history, and the current best hyperparameters.


In [ ]:
!pip -q install -U pip
!pip -q install tensorflow==2.20.0 cloudpickle matplotlib numpy pandas scikit-learn scipy seaborn pydantic optuna

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/hhihn/FewShotPainAdaptation.git"
PROJECT_DIR = Path("/content/FewShotPainAdaptation")

if not PROJECT_DIR.exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull

%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
from pathlib import Path
import shutil
import tarfile
import time

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/PainData")
BIOVID_TAR_CANDIDATES = [
    DRIVE_DATA_DIR / "BioVid.tar.gz",
    DRIVE_DATA_DIR / "biovid_parta.tar.gz",
    DRIVE_DATA_DIR / "BioVid.tgz",
    DRIVE_DATA_DIR / "biovid_parta.tgz",
    Path("/content/drive/MyDrive/BioVid.tar.gz"),
    Path("/content/drive/MyDrive/biovid_parta.tar.gz"),
    Path("/content/drive/MyDrive/BioVid.tgz"),
    Path("/content/drive/MyDrive/biovid_parta.tgz"),
]
BIOVID_TAR = next((path for path in BIOVID_TAR_CANDIDATES if path.exists()), None)
LOCAL_DATA_DIR = Path("/content/PainData")
LOCAL_BIOVID_DIR = LOCAL_DATA_DIR / "BioVid"
LOCAL_PARTA_DIR = LOCAL_BIOVID_DIR / "PartA"


def _has_biovid_parta(path: Path) -> bool:
    return (path / "Train").is_dir() and (path / "Test").is_dir()


def _find_local_biovid_root() -> Path | None:
    candidates = [
        LOCAL_DATA_DIR / "BioVid" / "PartA",
        LOCAL_DATA_DIR / "PartA",
        LOCAL_DATA_DIR,
    ]
    return next((path for path in candidates if _has_biovid_parta(path)), None)


def _safe_extract(tar: tarfile.TarFile, target_dir: Path) -> None:
    target_dir = target_dir.resolve()
    for member in tar.getmembers():
        member_path = (target_dir / member.name).resolve()
        if member_path != target_dir and target_dir not in member_path.parents:
            raise RuntimeError(f"Unsafe tar member path: {member.name}")
    tar.extractall(target_dir)


def _remove_metadata_files(root: Path) -> int:
    patterns = ("._*", ".DS_Store")
    removed = 0
    for pattern in patterns:
        for path in root.rglob(pattern):
            if path.is_file():
                path.unlink()
                removed += 1
    return removed


if BIOVID_TAR is None:
    searched = ", ".join(str(path) for path in BIOVID_TAR_CANDIDATES)
    raise FileNotFoundError(
        f"Could not find BioVid PartA tar archive. Searched: {searched}"
    )

if _find_local_biovid_root() is None:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    local_tar = Path("/content") / BIOVID_TAR.name
    if not local_tar.exists() or local_tar.stat().st_size != BIOVID_TAR.stat().st_size:
        print(f"Copying {BIOVID_TAR} to {local_tar} ...")
        start = time.perf_counter()
        shutil.copy2(BIOVID_TAR, local_tar)
        print(f"Copied in {time.perf_counter() - start:.1f}s")
    else:
        print(f"Using existing local archive: {local_tar}")

    print(f"Extracting to {LOCAL_DATA_DIR} ...")
    start = time.perf_counter()
    with tarfile.open(local_tar, "r:*") as tar:
        _safe_extract(tar, LOCAL_DATA_DIR)
    print(f"Extracted in {time.perf_counter() - start:.1f}s")
else:
    print(f"Using existing local BioVid directory: {_find_local_biovid_root()}")

removed_metadata_files = _remove_metadata_files(LOCAL_DATA_DIR)
if removed_metadata_files:
    print(f"Removed {removed_metadata_files} archive metadata files")

LOCAL_BIOVID_ROOT = _find_local_biovid_root()
if LOCAL_BIOVID_ROOT is None:
    raise FileNotFoundError(
        f"Archive did not produce expected Train/Test folders under {LOCAL_DATA_DIR}. "
        "Expected one of: BioVid/PartA/Train, PartA/Train, or Train. "
        "Create it with: tar -C /content/drive/MyDrive/PainData/BioVid -czf biovid_parta.tar.gz PartA"
    )

DATA_DIR = LOCAL_DATA_DIR
print(f"BioVid root: {LOCAL_BIOVID_ROOT}")
print(f"Training DATA_DIR: {DATA_DIR}")

In [ ]:
from argparse import Namespace
from datetime import datetime, timezone
import gc
import json
import logging
from pathlib import Path
import random
import time

import numpy as np
import optuna
import pandas as pd
import tensorflow as tf
from tensorflow.keras import mixed_precision

from tests.full_loso_trial import run_full_loso_trial
from utils.logger import setup_logger

# Colab search runs are GPU-oriented. Disable this check only for syntax/debug work.
REQUIRE_GPU = True
gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", gpus)
if REQUIRE_GPU and not gpus:
    raise RuntimeError(
        "No GPU detected. In Colab, use Runtime > Change runtime type > GPU."
    )

mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

ENABLE_DETERMINISM = False
SEED = 42
if ENABLE_DETERMINISM:
    try:
        tf.config.experimental.enable_op_determinism()
        print("Enabled deterministic TensorFlow ops")
    except Exception as exc:
        print("Deterministic ops not available:", exc)
else:
    print("Deterministic ops disabled for throughput")

tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/PainData")
DATA_DIR = Path(globals().get("LOCAL_DATA_DIR", DRIVE_DATA_DIR))
BIOVID_ROOT = globals().get("LOCAL_BIOVID_ROOT")
RUN_ROOT = Path("/content/drive/MyDrive/FewShotPainAdaptationHparamSearches")
DATASET_SOURCE = "biovid_part_a"

# Search budget. Increase N_TRIALS and N_LOSO_STEPS for the final search.
N_TRIALS = 24
N_LOSO_STEPS = 5
LOSO_START_INDEX = 1
LOSO_STOP_INDEX = LOSO_START_INDEX + N_LOSO_STEPS - 1
TIMEOUT_SECONDS = None

# Current CrossMod/CAN runs report source-subject prototype voting after training.
HELDOUT_METRIC = "source_subject_prototype_vote_accuracy"
STUDY_NAME = f"loso_hparam_{DATASET_SOURCE}_{HELDOUT_METRIC}"

# To resume a previous Drive-backed search after a Colab disconnect, set this to
# that run directory before executing this cell. Leave as None for a fresh run.
RESUME_RUN_DIR = None
# RESUME_RUN_DIR = Path("/content/drive/MyDrive/FewShotPainAdaptationHparamSearches/...")


def _has_biovid_parta(path: Path) -> bool:
    return (path / "Train").is_dir() and (path / "Test").is_dir()


def _resolve_biovid_root(
    data_dir: Path, staged_root: Path | None = None
) -> Path | None:
    candidates = []
    if staged_root is not None:
        candidates.append(Path(staged_root))
    candidates.extend(
        [
            data_dir / "BioVid" / "PartA",
            data_dir / "PartA",
            data_dir,
        ]
    )
    return next((path for path in candidates if _has_biovid_parta(path)), None)


if DATASET_SOURCE == "biovid_part_a":
    BIOVID_ROOT = _resolve_biovid_root(DATA_DIR, BIOVID_ROOT)
    if BIOVID_ROOT is None:
        raise FileNotFoundError(
            f"Missing BioVid PartA Train/Test folders under DATA_DIR={DATA_DIR}. "
            "Expected one of: BioVid/PartA, PartA, or DATA_DIR itself."
        )
else:
    required = ["X_pre.npy", "y_heater.npy", "subjects.npy"]
    missing = [name for name in required if not (DATA_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files in DATA_DIR={DATA_DIR}: {missing}")

if RESUME_RUN_DIR is None:
    run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    RUN_DIR = RUN_ROOT / f"{STUDY_NAME}-{run_stamp}"
else:
    RUN_DIR = Path(RESUME_RUN_DIR)
TRIAL_ROOT = RUN_DIR / "trials"
STUDY_DB = RUN_DIR / "study.sqlite3"
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)

logger = setup_logger("colab_loso_hparam_search", level=logging.INFO)
file_handler = logging.FileHandler(RUN_DIR / "hparam_search.log")
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s:%(lineno)d | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
logger.addHandler(file_handler)
logger.info("Search run directory: %s", RUN_DIR)
logger.info("Data directory: %s", DATA_DIR)
logger.info("BioVid root: %s", BIOVID_ROOT)
logger.info("Study database: %s", STUDY_DB)

BASE_DEFAULTS = {
    "dataset_source": DATASET_SOURCE,
    "data_variant": "real",
    "seed": SEED,
    "k_shot": 20,
    "q_query": 20,
    "task_class_ids": "0,4",
    "task_construction_mode": "mixed",
    "normalize_mode": "support",
    "attention_mode": "can",
    "can_attention_temperature": 0.025,
    "can_meta_hidden_dim": 32,
    "can_local_loss_weight": 1.0,
    "can_margin_loss_weight": 0.5,
    "can_margin_target": 0.5,
    "can_support_mode": "learned_prototype_memory",
    "learned_prototype_slots_per_class": 2,
    "prototype_bank_init_samples_per_class": 256,
    "prototype_finetune_epochs": 0,
    "prototype_finetune_tasks_per_epoch": 500,
    "prototype_phase2_loss_mode": "ce_can",
    "source_subject_prototype_vote_enabled": True,
    "source_subject_prototype_vote_use_base_index": True,
    "source_subject_prototype_vote_query_normalize_with_subject_stats": True,
    "source_subject_prototype_vote_softmax_scope": "global",
    "learning_rate": 6e-4,
    "lr_schedule": "cosine",
    "lr_decay_alpha": 0.1,
    "encoder_backend": "crossmod",
    "eegnet_temporal_filters": 8,
    "eegnet_depth_multiplier": 2,
    "eegnet_separable_filters": 16,
    "eegnet_temporal_kernel_size": 64,
    "eegnet_separable_kernel_size": 16,
    "eegnet_pool_size_1": 4,
    "eegnet_pool_size_2": 8,
    "eegnet_dropout_rate": 0.25,
    "eegnet_l2_weight": 1e-4,
    "crossmod_num_heads": 8,
    "crossmod_hidden_dim": 128,
    "crossmod_num_layers": 2,
    "crossmod_positional_base": 10000.0,
    "crossmod_attention_dropout_rate": 0.25,
    "crossmod_ff_activation": "relu",
    "gaussian_noise_std": 0.01,
    "deterministic_ops": ENABLE_DETERMINISM,
    "num_epochs": 1,
    "tasks_per_epoch": 10_000,
    "task_batch_size": 16,
    "task_chunk_size": 16,
    "val_tasks": 50,
    "heldout_eval_tasks": 500,
    "subject_eval_tasks": None,
    "k_shot_adaptation_steps": 0,
    "train_log_every": 125,
    "eval_log_every": 125,
    "val_batch_size": 50,
    "val_every_n_train_steps": 125,
    "validation_checkpoint_metric": "f1",
    "validation_checkpoint_mode": "auto",
    "summary_every_n_train_steps": 100,
    "train_prefetch_batches": 2,
    "train_progress_write_every_n_batches": 10,
    "csv_flush_every_events": 100,
    "disable_window_shift": False,
    "logging_verbosity": 1,
    "max_folds": None,
    "loso_start_index": LOSO_START_INDEX,
    "loso_stop_index": LOSO_STOP_INDEX,
}


In [ ]:
def _utc_now_iso() -> str:
    return datetime.now(tz=timezone.utc).isoformat()


def _json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, Namespace):
        return {key: _json_safe(val) for key, val in vars(value).items()}
    if isinstance(value, dict):
        return {str(key): _json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def _write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(_json_safe(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


SEARCH_SPACE = {
    "learning_rate": {
        "type": "float_log",
        "range": [1e-4, 1.5e-3],
        "baseline": BASE_DEFAULTS["learning_rate"],
        "reason": "Optimizer scale around the current 6e-4 CrossMod default.",
    },
    "task_batch_size": {
        "type": "categorical",
        "choices": [8, 16, 32],
        "baseline": BASE_DEFAULTS["task_batch_size"],
        "reason": "Controls episodic gradient batch size and GPU memory use.",
    },
    "lr_decay_alpha": {
        "type": "categorical",
        "choices": [0.05, 0.1, 0.2],
        "baseline": BASE_DEFAULTS["lr_decay_alpha"],
        "reason": "Final LR fraction for the fixed cosine schedule.",
    },
    "eegnet_temporal_filters": {
        "type": "categorical",
        "choices": [8, 12, 16, 24],
        "baseline": BASE_DEFAULTS["eegnet_temporal_filters"],
        "reason": "Per-modality temporal frontend width before CrossMod fusion.",
    },
    "eegnet_depth_multiplier": {
        "type": "categorical",
        "choices": [1, 2, 3],
        "baseline": BASE_DEFAULTS["eegnet_depth_multiplier"],
        "reason": "Depthwise sensor/frontend mixing capacity.",
    },
    "eegnet_separable_filters": {
        "type": "categorical",
        "choices": [16, 32, 64],
        "baseline": BASE_DEFAULTS["eegnet_separable_filters"],
        "reason": "CrossMod feature-map channel width; all choices divide selected heads.",
    },
    "eegnet_temporal_kernel_size": {
        "type": "categorical",
        "choices": [32, 64, 128],
        "baseline": BASE_DEFAULTS["eegnet_temporal_kernel_size"],
        "reason": "Temporal receptive field in each modality frontend.",
    },
    "eegnet_separable_kernel_size": {
        "type": "categorical",
        "choices": [8, 16, 32],
        "baseline": BASE_DEFAULTS["eegnet_separable_kernel_size"],
        "reason": "Temporal smoothing in each separable frontend block.",
    },
    "eegnet_pool_size_1": {
        "type": "categorical",
        "choices": [2, 4],
        "baseline": BASE_DEFAULTS["eegnet_pool_size_1"],
        "reason": "Early temporal downsampling rate.",
    },
    "eegnet_pool_size_2": {
        "type": "categorical",
        "choices": [4, 8],
        "baseline": BASE_DEFAULTS["eegnet_pool_size_2"],
        "reason": "Late temporal downsampling rate.",
    },
    "crossmod_num_heads": {
        "type": "categorical",
        "choices": [4, 8],
        "baseline": BASE_DEFAULTS["crossmod_num_heads"],
        "reason": "Transformer and cross-attention head count.",
    },
    "crossmod_hidden_dim": {
        "type": "categorical",
        "choices": [64, 128, 256],
        "baseline": BASE_DEFAULTS["crossmod_hidden_dim"],
        "reason": "Feed-forward width in CrossMod transformer layers.",
    },
    "crossmod_num_layers": {
        "type": "categorical",
        "choices": [1, 2, 3],
        "baseline": BASE_DEFAULTS["crossmod_num_layers"],
        "reason": "Depth of each modality transformer tower.",
    },
    "crossmod_attention_dropout_rate": {
        "type": "categorical",
        "choices": [0.0, 0.1, 0.25, 0.35],
        "baseline": BASE_DEFAULTS["crossmod_attention_dropout_rate"],
        "reason": "Attention and feed-forward dropout inside CrossMod blocks.",
    },
    "crossmod_ff_activation": {
        "type": "categorical",
        "choices": ["relu", "gelu"],
        "baseline": BASE_DEFAULTS["crossmod_ff_activation"],
        "reason": "Feed-forward nonlinearity for CrossMod transformer blocks.",
    },
    "can_attention_temperature": {
        "type": "categorical",
        "choices": [0.01, 0.025, 0.05, 0.1, 0.2],
        "baseline": BASE_DEFAULTS["can_attention_temperature"],
        "reason": "CAN prototype-query attention sharpness.",
    },
    "can_meta_hidden_dim": {
        "type": "categorical",
        "choices": [32, 64, 128],
        "baseline": BASE_DEFAULTS["can_meta_hidden_dim"],
        "reason": "CAN pair-scoring MLP width.",
    },
    "can_local_loss_weight": {
        "type": "categorical",
        "choices": [0.5, 1.0, 1.5],
        "baseline": BASE_DEFAULTS["can_local_loss_weight"],
        "reason": "Weight for CAN local temporal supervision.",
    },
    "can_margin_loss_weight": {
        "type": "categorical",
        "choices": [0.0, 0.25, 0.5, 0.75, 1.0],
        "baseline": BASE_DEFAULTS["can_margin_loss_weight"],
        "reason": "Weight for explicit true-class versus best-other CAN margin pressure.",
    },
    "can_margin_target": {
        "type": "categorical",
        "choices": [0.3, 0.5, 0.75, 1.0],
        "baseline": BASE_DEFAULTS["can_margin_target"],
        "reason": "Target CAN score separation for the margin loss.",
    },
    "learned_prototype_slots_per_class": {
        "type": "categorical",
        "choices": [1, 2, 5, 10],
        "baseline": BASE_DEFAULTS["learned_prototype_slots_per_class"],
        "reason": "Prototype-memory capacity per class.",
    },
    "prototype_bank_init_samples_per_class": {
        "type": "categorical",
        "choices": [0, 128, 256, 512, 870],
        "baseline": BASE_DEFAULTS["prototype_bank_init_samples_per_class"],
        "reason": "Number of source samples used to initialize prototype memory.",
    },
    "prototype_finetune_epochs": {
        "type": "categorical",
        "choices": [0, 1, 2],
        "baseline": BASE_DEFAULTS["prototype_finetune_epochs"],
        "reason": "Optional phase-2 updates of prototype-memory variables only.",
    },
    "prototype_finetune_tasks_per_epoch": {
        "type": "categorical",
        "choices": [250, 500, 1000],
        "baseline": BASE_DEFAULTS["prototype_finetune_tasks_per_epoch"],
        "condition": "prototype_finetune_epochs > 0",
        "reason": "Phase-2 task budget when prototype fine-tuning is enabled.",
    },
    "eegnet_dropout_rate": {
        "type": "categorical",
        "choices": [0.10, 0.25, 0.35, 0.45],
        "baseline": BASE_DEFAULTS["eegnet_dropout_rate"],
        "reason": "Regularization strength for per-modality frontends.",
    },
    "eegnet_l2_weight": {
        "type": "categorical",
        "choices": [1e-5, 3e-5, 1e-4, 3e-4, 1e-3],
        "baseline": BASE_DEFAULTS["eegnet_l2_weight"],
        "reason": "Kernel regularization for EEGNet-style frontend layers.",
    },
    "gaussian_noise_std": {
        "type": "categorical",
        "choices": [0.0, 0.005, 0.01, 0.02],
        "baseline": BASE_DEFAULTS["gaussian_noise_std"],
        "reason": "Light training-only signal augmentation for held-out subject robustness.",
    },
}

FIXED_SEARCH_SETTINGS = {
    "dataset_source": BASE_DEFAULTS["dataset_source"],
    "data_variant": BASE_DEFAULTS["data_variant"],
    "k_shot": BASE_DEFAULTS["k_shot"],
    "q_query": BASE_DEFAULTS["q_query"],
    "task_class_ids": BASE_DEFAULTS["task_class_ids"],
    "task_construction_mode": BASE_DEFAULTS["task_construction_mode"],
    "normalize_mode": BASE_DEFAULTS["normalize_mode"],
    "attention_mode": BASE_DEFAULTS["attention_mode"],
    "encoder_backend": BASE_DEFAULTS["encoder_backend"],
    "can_support_mode": BASE_DEFAULTS["can_support_mode"],
    "source_subject_prototype_vote_enabled": BASE_DEFAULTS[
        "source_subject_prototype_vote_enabled"
    ],
    "source_subject_prototype_vote_softmax_scope": BASE_DEFAULTS[
        "source_subject_prototype_vote_softmax_scope"
    ],
    "lr_schedule": BASE_DEFAULTS["lr_schedule"],
    "tasks_per_epoch": BASE_DEFAULTS["tasks_per_epoch"],
    "k_shot_adaptation_steps": BASE_DEFAULTS["k_shot_adaptation_steps"],
    "val_tasks": BASE_DEFAULTS["val_tasks"],
    "heldout_eval_tasks": BASE_DEFAULTS["heldout_eval_tasks"],
    "val_every_n_train_steps": BASE_DEFAULTS["val_every_n_train_steps"],
    "validation_checkpoint_metric": BASE_DEFAULTS["validation_checkpoint_metric"],
}


def _suggest_param(trial, name: str, spec: dict):
    param_type = spec["type"]
    if param_type == "float_log":
        low, high = spec["range"]
        return trial.suggest_float(name, low, high, log=True)
    if param_type == "float":
        low, high = spec["range"]
        return trial.suggest_float(name, low, high)
    if param_type == "categorical":
        return trial.suggest_categorical(name, spec["choices"])
    raise ValueError(f"Unsupported search parameter type for {name}: {param_type}")


def _sample_trial_params(trial) -> dict:
    params = {}
    for name, spec in SEARCH_SPACE.items():
        if name == "prototype_finetune_tasks_per_epoch":
            if int(params.get("prototype_finetune_epochs", 0)) <= 0:
                continue
        params[name] = _suggest_param(trial, name, spec)
    return params


def _build_trial_args(params: dict, trial_dir: Path) -> Namespace:
    values = dict(BASE_DEFAULTS)
    values.update(params)
    if int(values["prototype_finetune_epochs"]) <= 0:
        values["prototype_finetune_tasks_per_epoch"] = BASE_DEFAULTS[
            "prototype_finetune_tasks_per_epoch"
        ]
    training_dir = trial_dir / "training_progress"
    training_dir.mkdir(parents=True, exist_ok=True)
    return Namespace(
        data_dir=str(DATA_DIR),
        dataset_source=values["dataset_source"],
        data_variant=values["data_variant"],
        seed=values["seed"],
        k_shot=values["k_shot"],
        q_query=values["q_query"],
        task_class_ids=values["task_class_ids"],
        task_construction_mode=values["task_construction_mode"],
        normalize_mode=values["normalize_mode"],
        attention_mode=values["attention_mode"],
        can_attention_temperature=values["can_attention_temperature"],
        can_meta_hidden_dim=values["can_meta_hidden_dim"],
        can_local_loss_weight=values["can_local_loss_weight"],
        can_margin_loss_weight=values["can_margin_loss_weight"],
        can_margin_target=values["can_margin_target"],
        can_support_mode=values["can_support_mode"],
        learned_prototype_slots_per_class=values[
            "learned_prototype_slots_per_class"
        ],
        prototype_bank_init_samples_per_class=values[
            "prototype_bank_init_samples_per_class"
        ],
        prototype_finetune_epochs=values["prototype_finetune_epochs"],
        prototype_finetune_tasks_per_epoch=values[
            "prototype_finetune_tasks_per_epoch"
        ],
        prototype_phase2_loss_mode=values["prototype_phase2_loss_mode"],
        source_subject_prototype_vote_enabled=values[
            "source_subject_prototype_vote_enabled"
        ],
        source_subject_prototype_vote_use_base_index=values[
            "source_subject_prototype_vote_use_base_index"
        ],
        source_subject_prototype_vote_query_normalize_with_subject_stats=values[
            "source_subject_prototype_vote_query_normalize_with_subject_stats"
        ],
        source_subject_prototype_vote_softmax_scope=values[
            "source_subject_prototype_vote_softmax_scope"
        ],
        learning_rate=values["learning_rate"],
        lr_schedule=values["lr_schedule"],
        lr_decay_alpha=values["lr_decay_alpha"],
        encoder_backend=values["encoder_backend"],
        eegnet_temporal_filters=values["eegnet_temporal_filters"],
        eegnet_depth_multiplier=values["eegnet_depth_multiplier"],
        eegnet_separable_filters=values["eegnet_separable_filters"],
        eegnet_temporal_kernel_size=values["eegnet_temporal_kernel_size"],
        eegnet_separable_kernel_size=values["eegnet_separable_kernel_size"],
        eegnet_pool_size_1=values["eegnet_pool_size_1"],
        eegnet_pool_size_2=values["eegnet_pool_size_2"],
        eegnet_dropout_rate=values["eegnet_dropout_rate"],
        eegnet_l2_weight=values["eegnet_l2_weight"],
        crossmod_num_heads=values["crossmod_num_heads"],
        crossmod_hidden_dim=values["crossmod_hidden_dim"],
        crossmod_num_layers=values["crossmod_num_layers"],
        crossmod_positional_base=values["crossmod_positional_base"],
        crossmod_attention_dropout_rate=values[
            "crossmod_attention_dropout_rate"
        ],
        crossmod_ff_activation=values["crossmod_ff_activation"],
        gaussian_noise_std=values["gaussian_noise_std"],
        deterministic_ops=values["deterministic_ops"],
        num_epochs=values["num_epochs"],
        tasks_per_epoch=values["tasks_per_epoch"],
        task_batch_size=values["task_batch_size"],
        task_chunk_size=values["task_chunk_size"],
        val_tasks=values["val_tasks"],
        heldout_eval_tasks=values["heldout_eval_tasks"],
        subject_eval_tasks=values["subject_eval_tasks"],
        k_shot_adaptation_steps=values["k_shot_adaptation_steps"],
        train_log_every=values["train_log_every"],
        eval_log_every=values["eval_log_every"],
        val_batch_size=values["val_batch_size"],
        val_every_n_train_steps=values["val_every_n_train_steps"],
        validation_checkpoint_metric=values["validation_checkpoint_metric"],
        validation_checkpoint_mode=values["validation_checkpoint_mode"],
        summary_every_n_train_steps=values["summary_every_n_train_steps"],
        train_prefetch_batches=values["train_prefetch_batches"],
        train_progress_write_every_n_batches=values[
            "train_progress_write_every_n_batches"
        ],
        csv_flush_every_events=values["csv_flush_every_events"],
        disable_window_shift=values["disable_window_shift"],
        logging_verbosity=values["logging_verbosity"],
        training_progress_output_dir=str(training_dir),
        model_architecture_output=str(trial_dir / "model_summary.txt"),
        skip_model_architecture_save=False,
        max_folds=values["max_folds"],
        loso_start_index=values["loso_start_index"],
        loso_stop_index=values["loso_stop_index"],
        output_json=str(trial_dir / "full_loso_results.json"),
    )


def _metric_from_result(result: dict) -> float:
    if HELDOUT_METRIC not in result["summary"]:
        raise KeyError(
            f"Unknown HELDOUT_METRIC={HELDOUT_METRIC!r}; available: "
            f"{sorted(result['summary'].keys())}"
        )
    return float(result["summary"][HELDOUT_METRIC]["mean"])


In [ ]:
def _trial_to_record(trial) -> dict:
    duration_seconds = (
        float(trial.duration.total_seconds()) if trial.duration is not None else None
    )
    return {
        "trial_number": int(trial.number),
        "state": trial.state.name,
        "value": float(trial.value) if trial.value is not None else None,
        "params": dict(trial.params),
        "user_attrs": dict(trial.user_attrs),
        "datetime_start": (
            trial.datetime_start.isoformat()
            if trial.datetime_start is not None
            else None
        ),
        "datetime_complete": (
            trial.datetime_complete.isoformat()
            if trial.datetime_complete is not None
            else None
        ),
        "duration_seconds": duration_seconds,
    }


def _persist_study(study) -> None:
    records = [_trial_to_record(trial) for trial in study.trials]
    history_payload = {
        "updated_at_utc": _utc_now_iso(),
        "study_name": study.study_name,
        "direction": "maximize",
        "objective": HELDOUT_METRIC,
        "n_loso_steps": N_LOSO_STEPS,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "trials": records,
    }
    _write_json(RUN_DIR / "study_history.json", history_payload)

    rows = []
    for record in records:
        row = {
            "trial_number": record["trial_number"],
            "state": record["state"],
            "value": record["value"],
            "duration_seconds": record["duration_seconds"],
            "datetime_start": record["datetime_start"],
            "datetime_complete": record["datetime_complete"],
        }
        row.update({f"param_{key}": val for key, val in record["params"].items()})
        row.update({f"attr_{key}": val for key, val in record["user_attrs"].items()})
        rows.append(row)
    pd.DataFrame(rows).to_csv(RUN_DIR / "trials.csv", index=False)

    complete_trials = [
        trial
        for trial in study.trials
        if trial.state.name == "COMPLETE" and trial.value is not None
    ]
    if not complete_trials:
        return

    best_trial = max(complete_trials, key=lambda trial: float(trial.value))
    best_payload = {
        "updated_at_utc": _utc_now_iso(),
        "study_name": study.study_name,
        "objective": HELDOUT_METRIC,
        "direction": "maximize",
        "n_loso_steps": N_LOSO_STEPS,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "best_trial_number": int(best_trial.number),
        "best_value": float(best_trial.value),
        "best_params": dict(best_trial.params),
        "best_user_attrs": dict(best_trial.user_attrs),
    }
    _write_json(RUN_DIR / "best_hyperparameters.json", best_payload)


search_config = {
    "created_at_utc": _utc_now_iso(),
    "study_name": STUDY_NAME,
    "objective": HELDOUT_METRIC,
    "direction": "maximize",
    "n_trials": N_TRIALS,
    "timeout_seconds": TIMEOUT_SECONDS,
    "n_loso_steps": N_LOSO_STEPS,
    "loso_start_index": LOSO_START_INDEX,
    "loso_stop_index": LOSO_STOP_INDEX,
    "data_dir": DATA_DIR,
    "biovid_root": BIOVID_ROOT,
    "run_dir": RUN_DIR,
    "study_db": STUDY_DB,
    "base_defaults": BASE_DEFAULTS,
    "search_space": SEARCH_SPACE,
    "fixed_search_settings": FIXED_SEARCH_SETTINGS,
}
_write_json(RUN_DIR / "search_config.json", search_config)

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=min(8, max(1, N_TRIALS // 2)),
    multivariate=True,
    group=True,
)
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
    direction="maximize",
    sampler=sampler,
)

BASELINE_TRIAL = {
    key: spec["baseline"]
    for key, spec in SEARCH_SPACE.items()
    if key != "prototype_finetune_tasks_per_epoch"
}
BIGGER_CROSSMOD_TRIAL = dict(BASELINE_TRIAL)
BIGGER_CROSSMOD_TRIAL.update(
    {
        "eegnet_temporal_filters": 16,
        "eegnet_separable_filters": 32,
        "crossmod_hidden_dim": 256,
        "crossmod_num_layers": 2,
        "crossmod_attention_dropout_rate": 0.25,
        "learned_prototype_slots_per_class": 5,
        "prototype_bank_init_samples_per_class": 512,
        "prototype_finetune_epochs": 1,
        "prototype_finetune_tasks_per_epoch": 500,
        "eegnet_l2_weight": 3e-4,
        "gaussian_noise_std": 0.005,
    }
)
if len(study.trials) == 0:
    study.enqueue_trial(BASELINE_TRIAL)
    study.enqueue_trial(BIGGER_CROSSMOD_TRIAL)
    logger.info("Enqueued current baseline and larger CrossMod initial trials.")

print("Run directory:", RUN_DIR)
print("Study database:", STUDY_DB)
print("Objective:", HELDOUT_METRIC)
print("Fixed tasks_per_epoch:", BASE_DEFAULTS["tasks_per_epoch"])
print("LOSO fold range:", LOSO_START_INDEX, "to", LOSO_STOP_INDEX)


In [ ]:
def _summary_mean(result: dict, metric_name: str) -> float:
    return float(result["summary"].get(metric_name, {}).get("mean", float("nan")))


def objective(trial) -> float:
    tf.keras.backend.clear_session()
    gc.collect()

    params = _sample_trial_params(trial)
    trial_dir = TRIAL_ROOT / f"trial_{trial.number:04d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    args = _build_trial_args(params=params, trial_dir=trial_dir)

    hyperparameter_payload = {
        "created_at_utc": _utc_now_iso(),
        "trial_number": int(trial.number),
        "objective": HELDOUT_METRIC,
        "n_loso_steps": N_LOSO_STEPS,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "params": params,
        "args": args,
        "base_defaults": BASE_DEFAULTS,
    }
    _write_json(
        trial_dir / "training_progress" / "trial_hyperparameters.json",
        hyperparameter_payload,
    )

    logger.info("[Trial %s] Starting with params=%s", trial.number, params)
    started = time.perf_counter()
    try:
        result = run_full_loso_trial(args)
        score = _metric_from_result(result)
        zero_shot_mean = _summary_mean(result, "zero_shot_accuracy")
        zero_shot_f1_mean = _summary_mean(result, "zero_shot_f1")
        k_shot_mean = _summary_mean(result, "k_shot_accuracy")
        k_shot_f1_mean = _summary_mean(result, "k_shot_f1")
        vote_mean = _summary_mean(result, "source_subject_prototype_vote_accuracy")
        vote_f1_mean = _summary_mean(result, "source_subject_prototype_vote_f1")
        elapsed_seconds = float(time.perf_counter() - started)

        trial.set_user_attr("trial_dir", str(trial_dir))
        trial.set_user_attr("output_json", str(args.output_json))
        trial.set_user_attr("objective_metric", HELDOUT_METRIC)
        trial.set_user_attr("objective_metric_mean", score)
        trial.set_user_attr("zero_shot_accuracy_mean", zero_shot_mean)
        trial.set_user_attr("zero_shot_f1_mean", zero_shot_f1_mean)
        trial.set_user_attr("k_shot_accuracy_mean", k_shot_mean)
        trial.set_user_attr("k_shot_f1_mean", k_shot_f1_mean)
        trial.set_user_attr("source_subject_prototype_vote_accuracy_mean", vote_mean)
        trial.set_user_attr("source_subject_prototype_vote_f1_mean", vote_f1_mean)
        trial.set_user_attr("elapsed_seconds", elapsed_seconds)
        trial.set_user_attr("num_folds", int(result["summary"]["num_folds"]))

        trial_summary = {
            "completed_at_utc": _utc_now_iso(),
            "trial_number": int(trial.number),
            "objective": HELDOUT_METRIC,
            "score": score,
            "zero_shot_accuracy_mean": zero_shot_mean,
            "zero_shot_f1_mean": zero_shot_f1_mean,
            "k_shot_accuracy_mean": k_shot_mean,
            "k_shot_f1_mean": k_shot_f1_mean,
            "source_subject_prototype_vote_accuracy_mean": vote_mean,
            "source_subject_prototype_vote_f1_mean": vote_f1_mean,
            "elapsed_seconds": elapsed_seconds,
            "params": params,
            "args": args,
            "result_summary": result["summary"],
            "result_config": result["config"],
        }
        _write_json(trial_dir / "trial_summary.json", trial_summary)
        logger.info(
            "[Trial %s] Complete: %s=%.4f vote=%.4f zero_shot=%.4f k_shot=%.4f elapsed=%.1fs",
            trial.number,
            HELDOUT_METRIC,
            score,
            vote_mean,
            zero_shot_mean,
            k_shot_mean,
            elapsed_seconds,
        )
        return score
    except Exception as exc:
        failure_payload = {
            "failed_at_utc": _utc_now_iso(),
            "trial_number": int(trial.number),
            "params": params,
            "args": args,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }
        _write_json(trial_dir / "trial_failure.json", failure_payload)
        logger.exception("[Trial %s] Failed", trial.number)
        raise
    finally:
        tf.keras.backend.clear_session()
        gc.collect()


def after_trial(study_obj, trial_obj) -> None:
    _persist_study(study_obj)
    logger.info(
        "[Trial %s] state=%s value=%s",
        trial_obj.number,
        trial_obj.state.name,
        trial_obj.value,
    )


In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=TIMEOUT_SECONDS,
    callbacks=[after_trial],
    gc_after_trial=True,
    show_progress_bar=False,
)
_persist_study(study)

print("Search complete")
print("Run directory:", RUN_DIR)
complete_trials = [trial for trial in study.trials if trial.value is not None]
if complete_trials:
    print("Best value:", study.best_value)
    print("Best params:", study.best_params)
else:
    print("No completed trials yet.")

trials_csv = RUN_DIR / "trials.csv"
if trials_csv.exists():
    trials_frame = pd.read_csv(trials_csv)
    if "value" in trials_frame.columns:
        display(trials_frame.sort_values("value", ascending=False).head(10))
    else:
        display(trials_frame.head(10))
else:
    print("No trials.csv written yet.")

In [ ]:
best_path = RUN_DIR / "best_hyperparameters.json"
if best_path.exists():
    best_payload = json.loads(best_path.read_text(encoding="utf-8"))
    print(json.dumps(best_payload, indent=2, sort_keys=True))
else:
    print("No completed trials yet.")